<a href="https://colab.research.google.com/github/skyexry/urban-mobility-forecast/blob/main/notebooks/04_model_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!git clone https://github.com/skyexry/urban-mobility-forecast.git 2>/dev/null || git -C urban-mobility-forecast pull

Already up to date.


In [18]:
import sys
sys.path.append('/content/urban-mobility-forecast')

In [19]:
import sys
from google.colab import drive
drive.mount('/content/drive')

!pip install torch-geometric -q
sys.path.append('/content/urban-mobility-forecast')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
torch-geometric already in Drive, skipping install


In [ ]:
# repo already up to date via clone command above

In [21]:
from model.stconv import STConvBlock, build_edge_index

In [22]:
# Sanity check
batch, N, T = 4, 10, 72

x = torch.randn(batch, 1, N, T)
W = torch.rand(N, N)
W = (W + W.T) / 2
edge_index, edge_weight = build_edge_index(W)

block = STConvBlock(in_channels=1, hidden_channels=16, out_channels=32, kernel_size=3, K=3)
out = block(x, edge_index, edge_weight)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")  # (4, 32, 10, 68)

Input:  torch.Size([4, 1, 10, 72])
Output: torch.Size([4, 32, 10, 68])


## 1. TCN Block


In [23]:
# Sanity check
batch, time, features = 16, 72, 6
x = torch.randn(batch, features, time)

tcn = TCNBlock(in_channels=features, out_channels=64)
out = tcn(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")

Input:  torch.Size([16, 6, 72])
Output: torch.Size([16, 64, 72])


## 2. STCONV

In [28]:
# Sanity check
batch, N, T = 4, 10, 72

x = torch.randn(batch, 1, N, T)
W = torch.rand(N, N)
W = (W + W.T) / 2
L_hat = build_edge_index(W)

edge_index, edge_weight = build_edge_index(W)
block = STConvBlock(in_channels=1, hidden_channels=16, out_channels=32, kernel_size=3, K=3)
out = block(x, edge_index, edge_weight)

print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")  # (4, 32, 10, 68)

Input:  torch.Size([4, 1, 10, 72])
Output: torch.Size([4, 32, 10, 68])


## 3. STGNN

In [29]:
# Sanity check
batch, N, T = 4, 20, 72

x_demand = torch.randn(batch, N, T, 1)
x_time   = torch.randn(batch, T, 6)

W = torch.rand(N, N)
W = (W + W.T) / 2
edge_index, edge_weight = build_edge_index(W)

model = STGNN(num_nodes=N)
y_hat = model(x_demand, x_time, edge_index, edge_weight)
print(f"x_demand: {x_demand.shape}")
print(f"x_time:   {x_time.shape}")
print(f"y_hat:    {y_hat.shape}")  # (4, 20, 72)

x_demand: torch.Size([4, 20, 72, 1])
x_time:   torch.Size([4, 72, 6])
y_hat:    torch.Size([4, 20, 72])
